# Backdoor Finetuning: Multi-Model Sweep

Fine-tunes backdoored models on clean CIFAR-10 for 20 epochs.
Outputs per-model CSVs + merged CSV to Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted')


## 2. Clone Repo

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/alepelosi/backdoor_finetuning.git'
REPO_DIR = '/content/backdoor_finetuning'

if not Path(REPO_DIR).exists():
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
os.makedirs('data/cifar10', exist_ok=True)
os.makedirs('record', exist_ok=True)
print('Working dir:', os.getcwd())


## 3. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pyyaml', 'tqdm', 'pandas', 'matplotlib', 'scipy',
    'scikit-learn', 'scikit-image', 'opencv-python', 'pillow',
    'kornia', 'imageio', 'tensorboard', 'pytorch-wavelets'], check=True)
print('Dependencies installed')


## 4. Configure the Sweep

- `0_1` = 10% poison rate
- `0_01` = 1% poison rate

In [ ]:
import torch

DRIVE_MODELS_DIR = '/content/drive/MyDrive/models'
DRIVE_OUT_DIR    = '/content/drive/MyDrive/backdoor_results/finetuning_sweep'

MODEL_ZIPS = [
    'cifar10_preactresnet18_sig_0_1.zip',
    'cifar10_preactresnet18_bpp_0_1.zip',
    'cifar10_preactresnet18_inputaware_0_1.zip',
    'cifar10_preactresnet18_wanet_0_1.zip',
    'cifar10_preactresnet18_ssba_0_1.zip',
    'cifar10_preactresnet18_blended_0_1.zip',
    'cifar10_preactresnet18_lf_0_1.zip',
    'cifar10_preactresnet18_badnet_0_1.zip',
    'cifar10_preactresnet18_sig_0_01.zip',
    'cifar10_preactresnet18_bpp_0_01.zip',
    'cifar10_preactresnet18_inputaware_0_01.zip',
    'cifar10_preactresnet18_wanet_0_01.zip',
    'cifar10_preactresnet18_ssba_0_01.zip',
    'cifar10_preactresnet18_blended_0_01.zip',
    'cifar10_preactresnet18_lf_0_01.zip',
    'cifar10_preactresnet18_badnet_0_01.zip',
]

EPOCHS       = 20
LR           = 0.01
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
BATCH_SIZE   = 128
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_LABEL = 0

print('Device:', DEVICE)
print('Zips:', MODEL_ZIPS)


## 5. Patches & Imports

In [ ]:
import sys, os, re
from pathlib import Path

os.chdir('/content/backdoor_finetuning')
sys.path.insert(0, '/content/backdoor_finetuning')

import torch
import torch.serialization as _ts

_original_torch_load = _ts.load

def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_torch_load(*args, **kwargs)

torch.load = _safe_load

def _patch_numpy_compat():
    trainer = Path('/content/backdoor_finetuning/utils/trainer_cls.py')
    if trainer.exists():
        text = trainer.read_text()
        fixed = text.replace('np.infty', 'np.inf')
        if fixed != text:
            trainer.write_text(fixed)
            print('Patched np.infty -> np.inf')
_patch_numpy_compat()

def _replace_strings(obj, old, new):
    if isinstance(obj, str):
        return obj.replace(old, new)
    if isinstance(obj, dict):
        return {_replace_strings(k, old, new): _replace_strings(v, old, new)
                for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(_replace_strings(i, old, new) for i in obj)
    return obj

def fix_absolute_paths(local_dir):
    pt_path = Path(local_dir) / 'attack_result.pt'
    raw = pt_path.read_bytes()
    local_record = b'/content/backdoor_finetuning/record'

    if local_record in raw:
        print('  Paths already correct')
        return

    match = re.search(rb'(/[^\x00-\x1f\x80-\xff ]{5,}/record)', raw)
    if not match:
        print('  No absolute paths to patch')
        return

    old_prefix = match.group(1)
    print(f'  Found foreign path: {old_prefix.decode()}')

    if len(old_prefix) == len(local_record):
        pt_path.write_bytes(raw.replace(old_prefix, local_record))
        print('  Patched via direct replacement')
    else:
        data = _original_torch_load(str(pt_path), map_location='cpu', weights_only=False)
        data = _replace_strings(data, old_prefix.decode(), local_record.decode())
        torch.save(data, str(pt_path))
        print('  Patched via pickle rewrite')

from utils.save_load_attack import load_attack_result
from utils.aggregate_block.model_trainer_generate import generate_cls_model
print('Imports OK')


## 6. Evaluation Helpers

In [ ]:
def strip_module_prefix(state_dict):
    return {k[7:] if k.startswith('module.') else k: v for k, v in state_dict.items()}

def extract_state_dict(obj):
    if isinstance(obj, dict):
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj
        for key in ('model', 'model_state_dict', 'state_dict'):
            if key in obj:
                return extract_state_dict(obj[key])
    raise ValueError('Could not find a model state_dict in the checkpoint.')

def set_bd_original_label_mode(dataset, enabled):
    wrapped = getattr(dataset, 'wrapped_dataset', dataset)
    if not hasattr(wrapped, 'getitem_all_switch'):
        return lambda: None
    old = wrapped.getitem_all_switch
    wrapped.getitem_all_switch = enabled
    return lambda: setattr(wrapped, 'getitem_all_switch', old)

@torch.no_grad()
def evaluate(model, loader, target_label=None, device=DEVICE):
    model.eval()
    correct = total = 0
    for batch in loader:
        x, y = batch[0].to(device), batch[1].to(device)
        pred  = model(x).argmax(1)
        correct += (pred == target_label).sum().item() if target_label is not None                    else (pred == y).sum().item()
        total += y.size(0)
    return correct / total

def evaluate_suite(model, result, device=DEVICE):
    from torch.utils.data import DataLoader
    test_loader = DataLoader(result['clean_test'], batch_size=256,
                             shuffle=False, num_workers=2, pin_memory=True)
    bd_loader   = DataLoader(result['bd_test'],    batch_size=256,
                             shuffle=False, num_workers=2, pin_memory=True)
    acc = evaluate(model, test_loader)
    asr = evaluate(model, bd_loader, target_label=TARGET_LABEL)
    restore = set_bd_original_label_mode(result['bd_test'], True)
    ra = evaluate(model, bd_loader)
    restore()
    return {'ACC': acc, 'ASR': asr, 'RA': ra}


## 7. Fine-Tuning Function

In [ ]:
import torch.nn as nn, shutil, zipfile
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd

def run_finetuning(zip_name, drive_models_dir=DRIVE_MODELS_DIR):
    zip_path  = Path(drive_models_dir) / zip_name
    run_name  = zip_name.replace('.zip', '')
    local_dir = Path('/content/backdoor_finetuning/record') / run_name
    local_pt  = local_dir / 'attack_result.pt'

    if not zip_path.exists():
        raise FileNotFoundError(f'Zip not found: {zip_path}')

    # ── Unzip ────────────────────────────────────────────────────────────────
    local_dir.mkdir(parents=True, exist_ok=True)
    if not local_pt.exists():
        print(f'  Unzipping {zip_path} -> {local_dir} ...')
        with zipfile.ZipFile(str(zip_path), 'r') as zf:
            members  = zf.namelist()
            top_dirs = {m.split('/')[0] for m in members if '/' in m}
            has_root = (len(top_dirs) == 1 and
                        all(m.startswith(list(top_dirs)[0]) for m in members))
            if has_root:
                prefix = list(top_dirs)[0] + '/'
                for member in members:
                    rel = member[len(prefix):]
                    if not rel:
                        continue
                    dest = local_dir / rel
                    if member.endswith('/'):
                        dest.mkdir(parents=True, exist_ok=True)
                    else:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(member) as sf, open(dest, 'wb') as df_:
                            df_.write(sf.read())
            else:
                zf.extractall(str(local_dir))
        print('  Unzip complete.')
    else:
        print(f'  Using cached {local_dir}')

    fix_absolute_paths(local_dir)

    os.chdir('/content/backdoor_finetuning')

    print(f'  Loading {local_pt}')
    raw         = _original_torch_load(str(local_pt), map_location='cpu', weights_only=False)
    model_name  = raw.get('model_name', 'preactresnet18')
    num_classes = raw.get('num_classes', 10)

    result = load_attack_result(str(local_pt))
    model  = generate_cls_model(model_name, num_classes=num_classes)
    state  = strip_module_prefix(extract_state_dict(raw))
    model.load_state_dict(state)
    model  = model.to(DEVICE)

    train_loader = DataLoader(result['clean_train'], batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer,
                                                      milestones=[10, 15], gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    baseline = evaluate_suite(model, result)
    print(f'  Baseline | ACC {baseline["ACC"]:.4f} | ASR {baseline["ASR"]:.4f} | RA {baseline["RA"]:.4f}')

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = correct = total = 0
        for batch in train_loader:
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            correct      += (logits.detach().argmax(1) == y).sum().item()
            total        += x.size(0)
        scheduler.step()

        metrics = evaluate_suite(model, result)
        row = {'epoch': epoch, 'loss': running_loss / total, **metrics}
        history.append(row)
        print(f'  Epoch {epoch:02d} | loss {row["loss"]:.4f} | '
              f'ACC {row["ACC"]:.4f} | ASR {row["ASR"]:.4f} | RA {row["RA"]:.4f}')

    return pd.DataFrame(history), baseline, model, model_name, run_name


## 8. Run the Sweep

In [ ]:
import json as _json, shutil
from datetime import datetime

Path(DRIVE_OUT_DIR).mkdir(parents=True, exist_ok=True)

all_dfs = []
summary = []

for zip_name in MODEL_ZIPS:
    print()
    print('=' * 70)
    print(f'  {zip_name}')
    print('=' * 70)
    started = datetime.now().isoformat(timespec='seconds')
    try:
        df, baseline, model, model_name, run_name = run_finetuning(zip_name)

        run_dir = Path(DRIVE_OUT_DIR) / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        df.to_csv(run_dir / 'finetuning_history.csv', index=False, sep=';')
        torch.save(model.cpu().state_dict(), run_dir / 'finetuned_model.pt')

        meta = {
            'run_name': run_name, 'model': model_name,
            'baseline': baseline, 'final': df.iloc[-1].to_dict(),
            'settings': {'epochs': EPOCHS, 'lr': LR, 'momentum': MOMENTUM,
                         'weight_decay': WEIGHT_DECAY, 'batch_size': BATCH_SIZE,
                         'target_label': TARGET_LABEL},
            'started_at': started,
            'finished_at': datetime.now().isoformat(timespec='seconds'),
        }
        (run_dir / 'finetuning_metrics.json').write_text(_json.dumps(meta, indent=2))

        df_tagged = df.copy()
        df_tagged.insert(0, 'run_name', run_name)
        all_dfs.append(df_tagged)

        summary.append({'run_name': run_name, 'status': 'ok',
                        'baseline_ACC': baseline['ACC'], 'baseline_ASR': baseline['ASR'],
                        'final_ACC': df.iloc[-1]['ACC'], 'final_ASR': df.iloc[-1]['ASR'],
                        'final_RA':  df.iloc[-1]['RA']})
        print(f'  Saved to {run_dir}')

    except Exception as exc:
        import traceback; traceback.print_exc()
        summary.append({'run_name': zip_name, 'status': f'failed: {exc}',
                        'baseline_ACC': '', 'baseline_ASR': '',
                        'final_ACC': '', 'final_ASR': '', 'final_RA': ''})

if all_dfs:
    import pandas as pd
    merged = pd.concat(all_dfs, ignore_index=True)
    merged.to_csv(Path(DRIVE_OUT_DIR) / 'all_runs_finetuning_history.csv', index=False, sep=';')
    print('\nMerged CSV saved.')

import pandas as pd
summary_df = pd.DataFrame(summary)
summary_df.to_csv(Path(DRIVE_OUT_DIR) / 'sweep_summary.csv', index=False, sep=';')
print('\nSweep summary:')
print(summary_df.to_string(index=False))
